# Práctica 2 - Práctica de búsqueda

Mateo Fraguas Abal 

Un problema de búsqueda tiene un corpus $ C $, consultas $ q $ y un índice invertido $ I $ que los empareja de manera eficiente. Para cada consulta usamos un juicio de relevancia, un ground truth elaborado por expertos que evalúa la calidad del sistema. 

In [28]:
from bs4 import BeautifulSoup
from pathlib import Path 
from whoosh.index import create_in
from whoosh.fields import *

/home/mateo/Documentos/clase/ling/.venv/lib/python3.13/site-packages/whoosh/analysis/filters.py:56: SyntaxWarning: invalid escape sequence '\w'
  \w+([:.]?\w+)*         # word characters, with opt. internal colons/dots
/home/mateo/Documentos/clase/ling/.venv/lib/python3.13/site-packages/whoosh/analysis/filters.py:158: SyntaxWarning: invalid escape sequence '\S'
  >>> ana = RegexTokenizer(r"\S+") | TeeFilter(f1, f2)
/home/mateo/Documentos/clase/ling/.venv/lib/python3.13/site-packages/whoosh/analysis/intraword.py:49: SyntaxWarning: invalid escape sequence '\S'
  >>> analyzer = RegexTokenizer(r"\S+") | cwf
/home/mateo/Documentos/clase/ling/.venv/lib/python3.13/site-packages/whoosh/analysis/intraword.py:275: SyntaxWarning: invalid escape sequence '\S'
  >>> analyzer = RegexTokenizer(r"\S+") | iwf | LowercaseFilter()
/home/mateo/Documentos/clase/ling/.venv/lib/python3.13/site-packages/whoosh/analysis/intraword.py:285: SyntaxWarning: invalid escape sequence '\|'
  def __init__(self, delims=u

## Carga del corpus

In [44]:
ruta_cacm = Path('cacm')
archivos_html = sorted(ruta_cacm.glob('*.html'))

html_doc = archivos_html[0]
print(f"archivo: {html_doc.name}")
with html_doc.open() as f:

    soup = BeautifulSoup(f, 'html.parser')

# en este caso no hay ningún título, párrafo ni enlace
print(soup.title) # Nombre del título
print(soup.p) # Primer párrafo
print(soup.a) # Primer enlace
print(soup.find_all('a')) # Todos los enlaces
print(soup.find(id="link")) # Enlace con id="link"

# Sólohay texto 
print(soup.get_text())

archivo: CACM-0001.html
None
None
None
[]
None




Preliminary Report-International Algebraic Language

CACM December, 1958

Perlis, A. J.
Samelson,K.

CA581203 JB March 22, 1978  8:28 PM

100	5	1
123	5	1
164	5	1
1	5	1
1	5	1
1	5	1
205	5	1
210	5	1
214	5	1
1982	5	1
398	5	1
642	5	1
669	5	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
165	6	1
196	6	1
196	6	1
1273	6	1
1883	6	1
324	6	1
43	6	1
53	6	1
91	6	1
410	6	1
3184	6	1






Eliminamos caracteres vacíos y guardamos el título

In [51]:
# El texto tiene caracteres vacios al inicio (10: LF, 32: space)
##for chr in soup.get_text():
##    print(ord(chr))
    
# Eliminamos los valores 32 y 10 del principio y del final
# Tomamos el título, el texto antes del primer 10

# Guardamos todos los textos de los archivos
titulos = []
textos = []
for archivo in archivos_html:
    with archivo.open() as f:
        soup = BeautifulSoup(f, 'html.parser')
        
        # Eliminamos los valores 32 y 10 del principio y del final
        texto = soup.get_text()
        i = 0
        for t in texto:
            if ord(t) != 32 and ord(t) != 10:
                break
            i += 1
        texto = texto[i:]
        i = 0
        for t in texto:
            if ord(t) == 10:
                break
            i += 1
        titulo = texto[:i]
        texto = texto[i:]
        i = 0
        for t in texto:
            if ord(t) != 32 and ord(t) != 10:
                break
            i += 1
        texto = texto[i:]
        i = 0
        for t in reversed(texto):
            if ord(t) != 32 and ord(t) != 10:
                break
            i += 1
        texto = texto[:-i] if i > 0 else texto
        textos.append(texto)
        titulos.append(titulo)
        
print(titulos[0])
print(textos[0])

Preliminary Report-International Algebraic Language
CACM December, 1958

Perlis, A. J.
Samelson,K.

CA581203 JB March 22, 1978  8:28 PM

100	5	1
123	5	1
164	5	1
1	5	1
1	5	1
1	5	1
205	5	1
210	5	1
214	5	1
1982	5	1
398	5	1
642	5	1
669	5	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
1	6	1
165	6	1
196	6	1
196	6	1
1273	6	1
1883	6	1
324	6	1
43	6	1
53	6	1
91	6	1
410	6	1
3184	6	1


In [ ]:

print(f"Cantidad de textos: {len(textos)}")

Cantidad de textos: 3204


## Creación del indice

In [ ]:
schema = Schema(title=TEXT(stored=True), path=ID(stored=True), content=TEXT)
ix = create_in("indexdir", schema)
writer = ix.writer()
writer.add_document(title=u"First document", path=u"/a",content=u"This is the first document we've added!") writer.add_document(title=u"Second document", path=u"/b",content=u"The second one is even more interesting!")
writer.commit()
from whoosh.qparser import QueryParser
with ix.searcher() as searcher:
    query = QueryParser("content", ix.schema).parse("first")
    results = searcher.search(query)
    print(results[0])  